# TERRA on the in-vivo perturb-FISH dataset -- sanity checks

Before running TERRA (`lotfollahi-lab/TERRA-96M`) on `pfish_terra.h5ad`, check:

1. **Encoder coverage**: how many of the 500 gene symbols in `pfish_terra.h5ad` map to Ensembl IDs
   that TERRA's pretrained tokenizer actually knows (`ensembl_dictionary.pkl` for symbol -> Ensembl ID,
   `token_dictionary.pkl` for the tokenizer's ~22k-gene vocab). This mirrors `scripts/terra/common.py`'s
   `token_delta()` (symbol -> Ensembl -> token lookup).

In [7]:
import pickle

import h5py
import numpy as np
import scanpy as sc

PFISH_TERRA = "../../notebooks/in_vivo/pfish_terra.h5ad"   # 500 genes, full encoder-side panel
PFISH_FULL = "../../notebooks/in_vivo/pfish_full.h5ad"     # 154 genes, comparable to other models


def read_var_names(h5ad_path):
    # anndata.read_h5ad(..., backed="r") errors on this file's /uns/log1p (IORegistryError on
    # older anndata); var_names alone don't need anndata at all.
    with h5py.File(h5ad_path, "r") as f:
        idx = f["var"]["_index"][:]
    return [g.decode() if isinstance(g, bytes) else g for g in idx]


genes_500 = read_var_names(PFISH_TERRA)
genes_154 = read_var_names(PFISH_FULL)
print(f"pfish_terra.h5ad: {len(genes_500)} genes")
print(f"pfish_full.h5ad:  {len(genes_154)} genes")
assert set(genes_154) <= set(genes_500), "154-gene panel is not a subset of the 500-gene panel"
print("154-gene panel confirmed as a subset of the 500-gene panel.")

pfish_terra.h5ad: 500 genes
pfish_full.h5ad:  154 genes
154-gene panel confirmed as a subset of the 500-gene panel.


## 1. Encoder coverage of the 500 genes

Downloads the pretrained bundle (network access to the HF Hub required the first time; if
`lotfollahi-lab/TERRA-96M` is private you'll need `HF_TOKEN` set / `huggingface-cli login`).

In [10]:
from terra import download_pretrained

MODEL_REPO = "lotfollahi-lab/TERRA-96M"
LOCAL_DIR = "/data/a330d/projects/cellina-reproducibility/pretrained/"
model_dir = download_pretrained(MODEL_REPO, local_dir=LOCAL_DIR)
print("model bundle:", model_dir)

with open(f"{model_dir}/ensembl_dictionary.pkl", "rb") as f:
    ens_map = pickle.load(f)          # gene symbol -> Ensembl ID
with open(f"{model_dir}/token_dictionary.pkl", "rb") as f:
    token_dict = pickle.load(f)       # Ensembl ID (+ special tokens) -> token id

encoder_vocab = {k for k in token_dict if "ENS" in k}
print(f"tokenizer vocab: {len(encoder_vocab)} ENS-prefixed gene tokens (+ {len(token_dict) - len(encoder_vocab)} special tokens)")

/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import 

INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/lotfollahi-lab/TERRA-96M/revision/main "HTTP/1.1 200 OK"


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 874.05it/s]

model bundle: /data/a330d/projects/cellina-reproducibility/pretrained
tokenizer vocab: 21952 ENS-prefixed gene tokens (+ 1455 special tokens)


In [14]:
rows = []
for g in genes_500:
    ens_id = ens_map.get(g)
    rows.append((g, ens_id, (ens_id in encoder_vocab) if ens_id is not None else False))

no_ensembl = [g for g, e, ok in rows if e is None]
not_in_vocab = [g for g, e, ok in rows if e is not None and not ok]
covered = [g for g, e, ok in rows if ok]

print(f"{len(covered)}/{len(genes_500)} genes covered by TERRA's pretrained encoder vocab")
print(f"  {len(no_ensembl)} symbols have no Ensembl ID in ensembl_dictionary.pkl")
print(f"  {len(not_in_vocab)} symbols resolve to an Ensembl ID absent from token_dictionary.pkl")
if no_ensembl:
    print("no Ensembl mapping:", sorted(no_ensembl))
if not_in_vocab:
    print("Ensembl ID not tokenized:", sorted(not_in_vocab))

498/500 genes covered by TERRA's pretrained encoder vocab
  2 symbols have no Ensembl ID in ensembl_dictionary.pkl
  0 symbols resolve to an Ensembl ID absent from token_dictionary.pkl
no Ensembl mapping: ['TRAC', 'TRBC1']


## 2. LoRA fine-tune on `pfish_terra` (self-supervised, all cells)

Notebook version of `scripts/terra/finetuning_lora_pfish.py` -- same cells, same code, so you can
step through / inspect intermediate state instead of running it as a script. See that file's
docstring for the full rationale (px_to_um source, why this bypasses `common.load_dataset()`, the
`/uns/log1p` anndata workaround).

**Real run, CRC-parity settings**: `EPOCHS=5`, `BATCH_SIZE=128`, `LR=1e-4` -- identical to
`scripts/terra/finetuning_lora.py`'s Xenium-tutorial recipe (same LoRA config, same collapse guard).
`BATCH_SIZE=128` was empirically probed on this machine's RTX 4090 (24 GB) at **13.6 GiB peak**
(vs. CRC's 38.5 GiB on a 48 GB card -- pfish's panel is far sparser, max 221 nonzero genes/cell vs.
the 256-token cap), so it fits with comfortable headroom; kept at 128 rather than raised further to
stay faithful to the recipe LR was tuned against. All 154,416 cells that survive harmonization are
used (2 of 154,418 dropped by the `min_genes_per_cell=10` filter), no holdout -- same as CRC's
self-supervised fine-tune. Expect roughly 1,206 steps/epoch x 5 epochs; the collapse-guard step at
the end embeds a 20,000-cell subsample through all 6 checkpoints (frozen + 5 epochs).

In [18]:
import json
import logging
import os
import re
import shutil
import sys
import time
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("TQDM_DISABLE", "1")

import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import yaml
from datasets import load_from_disk

sys.path.insert(0, str(Path.cwd()))   # notebook cwd == scripts/terra, for `import common`
import common
import terra.training.finetune_self_supervised as fss
from terra.inference import embed_dataset, harmonize_adata
from terra.training.finetune_self_supervised import finetune_self_supervised, prepare_finetuned_model
from terra.utils.helper import init_model, parse_arch_kwargs, parse_protein_init_kwargs

# pfish_terra.h5ad has an /uns/log1p/base entry written with a "null" IOSpec encoding this anndata
# doesn't have a reader for -- harmless scanpy log1p-base marker, not read by anything below.
# Teach the registry to decode it as None instead of touching the file.
from anndata._io.specs.registry import _REGISTRY, IOSpec

if (h5py.Dataset, IOSpec("null", "0.1.0")) not in _REGISTRY.read:
    @_REGISTRY.register_read(h5py.Dataset, IOSpec("null", "0.1.0"))
    def _read_null(elem, _reader):
        return None

logging.basicConfig(level=logging.WARNING, format="%(message)s")
for name in ("terra.training.finetune_self_supervised", "terra.utils.helper"):
    logging.getLogger(name).setLevel(logging.INFO)

TARGETS = ["qkv", "proj", "fc1", "fc2"]   # LoRA-adapted submodules
EPS = 1e-6      # "changed" cannot be bitwise: EMA touches every tensor every step.

In [19]:
# --- config: real run, CRC-parity settings (set EPOCHS=1, MAX_STEPS=5, BATCH_SIZE=32 for a smoke test) ---
ADATA_PATH = PFISH_TERRA
PX_TO_UM = 0.108     # pfish_eda (1).ipynb: um_per_pixel = 5.4 / 50
EPOCHS = 5
MAX_STEPS = None     # caps cells to MAX_STEPS*BATCH_SIZE (1 epoch only) -- smoke tests only
BATCH_SIZE = 128      # probed: 13.6 GiB peak on this 24 GB RTX 4090, matches CRC's batch size
LR = 1e-4
SEED = 0
GUARD_CELLS = 20000
MAX_CELLS = None
NPROC = 16
FROM_RUN_DIR = None   # set to an existing run dir to skip training and only repack/embed/guard

SID = Path(ADATA_PATH).stem                                              # "pfish_terra"
# .resolve(): section 3's decoder training passes WORK-derived paths to a subprocess with a
# DIFFERENT cwd (repo root) -- a relative path here would resolve against the wrong directory there.
WORK = Path("../../notebooks/in_vivo/terra_pfish/terra").resolve()       # sibling to cellina_*_pfish dirs
RUN_DIR = WORK / "lora_run"
SELECTION = WORK / "epoch_selection.json"
TOK_CACHE = WORK.parent / "terra_tok"
RUN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path(model_dir)
CFG = yaml.safe_load(open(MODEL_DIR / "model_config.yaml"))
TOKD = token_dict   # already loaded above from token_dictionary.pkl

In [6]:
def build_encoder():
    """The encoder exactly as embed.py builds it from the bundle's model_config.yaml."""
    n_special_tokens = len(CFG["meta"]["special_tokens"])
    seq_len = (CFG["data"]["seq_len_cell"] + CFG["data"]["seq_len_neighborhood"]
               + n_special_tokens)
    enc, _ = init_model(
        gt_type=CFG["meta"]["gt_type"], count_encoding=CFG["meta"]["count_encoding"],
        n_value_bins=CFG["meta"]["n_value_bins"], cell_pos_enc=CFG["meta"]["cell_pos_enc"],
        device="cpu", vocab_size=len(TOKD), seq_len=seq_len,
        n_special_tokens=n_special_tokens, n_segments=CFG["data"]["n_segments"],
        enc_emb_dim=CFG["meta"]["enc_emb_dim"], enc_depth=CFG["meta"]["enc_depth"],
        pred_emb_dim=CFG["meta"]["pred_emb_dim"], pred_depth=CFG["meta"]["pred_depth"],
        num_heads=CFG["meta"]["num_heads"], mlp_ratio=CFG["meta"]["mlp_ratio"],
        use_flash_attention=CFG["meta"]["use_flash_attention"],
        api_version=CFG["meta"]["api_version"],
        sep_gene_tokens_neb=CFG["data"]["sep_gene_tokens_neb"],
        predict_gene=CFG["meta"]["predict_gene"], pos_learnable=CFG["meta"]["pos_learnable"],
        n_special_values=CFG["data"].get("n_special_values", 0),
        nz_spc=CFG["data"].get("nz_spc", False),
        mlp_bias=CFG["meta"].get("mlp_bias", True),
        protein_init_kwargs=parse_protein_init_kwargs(CFG, TOKD),
        **parse_arch_kwargs(CFG))
    return enc

In [ ]:
def load_pfish_terra(adata_path, model_dir, px_to_um):
    """Mirror common.py's _terra_side() crc branch: pfish gene symbols are already human, so no
    merfish-style upper-case/ortholog step is needed -- just harmonize_adata's own symbol lookup."""
    raw = sc.read_h5ad(adata_path)
    raw.obs_names_make_unique()
    raw.obs["cell_id"] = raw.obs_names.astype(str)

    coords = np.asarray(raw.obsm["spatial"], dtype=np.float64) * px_to_um
    raw.obsm["spatial"] = coords
    ext = np.ptp(coords, axis=0)
    print(f"[terra] tissue extent: {ext[0]:.0f} x {ext[1]:.0f} um (px_to_um={px_to_um})")

    if sp.issparse(raw.X):
        raw.X = raw.X.tocsr()
    raw.layers["counts"] = raw.X.copy()
    raw = harmonize_adata(
        raw,
        gene_mapping_dict_file_path=f"{model_dir}/ensembl_dictionary.pkl",
        gene_occurrence_count_file_path=f"{model_dir}/gene_count_dictionary.pkl",
    )
    if sp.issparse(raw.X):
        raw.X = raw.X.tocsr()
    raw.layers["counts"] = raw.X.copy()     # re-sync after harmonize's gene/cell filtering
    return raw


adata_terra = load_pfish_terra(ADATA_PATH, str(MODEL_DIR), PX_TO_UM)
print(f"[harmonize] {adata_terra.n_obs:,} cells x {adata_terra.n_vars} genes survive TERRA harmonization")

In [28]:
adata_terra

AnnData object with n_obs × n_vars = 154416 × 498
    obs: 'cell_index', 'area', 'total_counts', 'celltype', 'celltype2', 'perturbation', 'n_perturb', 'n_counts', 'cell_id', 'n_genes'
    var: 'ensembl_id', 'n_cells'
    uns: 'log1p'
    obsm: 'spatial'
    layers: 'counts', 'lognorm'

In [29]:
tok_all = common.tokenize_cached(adata_terra, str(MODEL_DIR), TOK_CACHE, nproc=NPROC)
del adata_terra

In [30]:
FT_TOK = TOK_CACHE
if MAX_STEPS:
    assert EPOCHS == 1, "MAX_STEPS implies a single epoch"
    MAX_CELLS = min(MAX_STEPS * BATCH_SIZE, MAX_CELLS or len(tok_all))
if MAX_CELLS and MAX_CELLS < len(tok_all):
    # ponytail: smoke only -- plain random subsample of rows; no cell-id filtering needed
    # because the self-supervised objective is label-free.
    FT_TOK = RUN_DIR / "tok_sub"
    if not FT_TOK.exists():
        rng = np.random.default_rng(SEED)
        idx = np.sort(rng.choice(len(tok_all), MAX_CELLS, replace=False))
        tok_all.select(idx).flatten_indices().save_to_disk(str(FT_TOK))
    tok_all = load_from_disk(str(FT_TOK))
N_CELLS = len(tok_all)
print(f"[tok] {N_CELLS} rows from {FT_TOK} | columns {list(tok_all.features)}", flush=True)

[tok] 154416 rows from /data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/terra_pfish/terra_tok | columns ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens']


In [10]:
# fss._build_model calls init_model without the config-dependent extras (nz_spc / n_special_values /
# protein_init / arch kwargs), same gap finetuning.py patches.
_orig_init_model = fss.init_model


def _patched_init_model(**kw):
    kw["nz_spc"] = CFG["data"].get("nz_spc", False)
    kw["mlp_bias"] = CFG["meta"].get("mlp_bias", True)
    kw["n_special_values"] = CFG["data"].get("n_special_values", 0)
    kw["protein_init_kwargs"] = parse_protein_init_kwargs(CFG, TOKD)
    kw.update(parse_arch_kwargs(CFG))
    return _orig_init_model(**kw)


fss.init_model = _patched_init_model

# fss.init_cell_dataset also omits `pad_special_tokens=True`, which embed_dataset always sets:
# with special tokens (112M: ["batch"]) the loader would otherwise look up a per-cell
# `batch_value` -- a corpus batch identity (`spv_{dataset_id}_{batch}`) that new slides cannot
# have.  Padding the slot is exactly what inference does, so train and embed see the same input.
_orig_init_cell_dataset = fss.init_cell_dataset


def _patched_init_cell_dataset(**kw):
    kw.setdefault("pad_special_tokens", True)
    kw.setdefault("truncate_neighbors", CFG["data"].get("truncate_neighbors", False))
    kw.setdefault("tokenized_seq_len_cell", CFG["data"].get("tokenized_seq_len_cell", None))
    return _orig_init_cell_dataset(**kw)


fss.init_cell_dataset = _patched_init_cell_dataset

In [11]:
# preflight: the pretrained target-encoder weights must load into build_encoder() with no key
# mismatch, before we spend time fine-tuning.
PRETRAINED = torch.load(MODEL_DIR / "model_checkpoint.pt", map_location="cpu")["target_encoder"]
PRETRAINED = {k.replace("module.", ""): v for k, v in PRETRAINED.items()}
_enc = build_encoder()
_enc.load_state_dict(PRETRAINED)     # strict; raises on any mismatch
n_total = sum(p.numel() for p in _enc.parameters())
print(f"[preflight] ALL KEYS MATCHED | {n_total:,} encoder params", flush=True)
del _enc

/tmp/ipykernel_1123859/649073396.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  PRETRAINED = torch.load(MODEL_DIR / "model_checkpoint.pt", map_location="cpu")["target_e

INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.
INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )


In [12]:
# log capture + training clock
epoch_log = []
_EPOCH_RE = re.compile(r"Epoch \[(\d+)/\d+\], Loss: ([\d.]+), LR: ([\d.e+-]+)")


class _EpochLog(logging.Handler):
    def emit(self, record):
        m = record.getMessage()
        g = _EPOCH_RE.search(m)
        if g:
            epoch_log.append({"line": m, "t": time.time(), "epoch": int(g[1]),
                              "loss": float(g[2]), "lr": float(g[3])})


logging.getLogger("terra.training.finetune_self_supervised").addHandler(_EpochLog())

# Start the clock at the first real step so encoder build / dataloader spin-up is
# excluded from sec_per_step. apply_masks runs once per step inside the loop.
first_step = []
_orig_apply_masks = fss.apply_masks


def _timed_apply_masks(*a, **kw):
    if not first_step:
        first_step.append(time.time())
    return _orig_apply_masks(*a, **kw)


fss.apply_masks = _timed_apply_masks

In [13]:
# fine-tune -- Xenium tutorial recipe except lr (see LR) and warmup (0.1 epoch: our epochs are
# ~50x longer than the tutorial's, so a 1-epoch warmup never reaches peak)
finetune_args = {
    "model": {"pretrained_checkpoint_path": str(MODEL_DIR),
              "finetune_checkpoint_path": str(RUN_DIR)},
    "data": {"finetune_dataset": [str(FT_TOK)], "batch_size": BATCH_SIZE,
             "num_workers": 8, "pin_memory": True, "drop_last": True,
             "sample_segments": False, "sample_gene_masks": True},
    "finetune": {"num_epochs": EPOCHS, "lr": LR, "start_lr": LR / 10, "final_lr": LR / 10,
                 "warmup_epochs": 0.1, "weight_decay": 0.04, "final_weight_decay": 0.4,
                 "ema_momentum": 0.9995, "final_ema_momentum": 1.0,
                 "loss_fn_type": "smooth_l1", "clip_grad": 2.0, "use_bfloat16": True,
                 "use_peft": True, "peft_method": "lora", "peft_rank": 16,
                 "peft_alpha": 256, "peft_dropout": 0.1, "peft_bias": "none",
                 "peft_target_modules": TARGETS, "save_every": 1},
}
steps_per_epoch = N_CELLS // BATCH_SIZE          # drop_last=True

if FROM_RUN_DIR:
    run_dir = Path(FROM_RUN_DIR)
    wall = peak_gib = float("nan")
    old = next(iter(sorted(run_dir.parents[1].glob("*bundle*/ft_summary.json"))), None)
    prev = json.loads(old.read_text()) if old else {}
    loss_per_epoch = prev.get("loss_per_epoch", [])
    steps_per_epoch = prev.get("steps_per_epoch", steps_per_epoch)
    print(f"[reuse] {run_dir} | prior summary: {old}", flush=True)
else:
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    run_dir = Path(finetune_self_supervised(args=finetune_args, save_folder_path=str(RUN_DIR),
                                            run_name="run"))
    wall = time.time() - t0
    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    assert len(epoch_log) == EPOCHS, f"{len(epoch_log)} epoch lines for {EPOCHS} epochs"
    loss_per_epoch = [e["loss"] for e in epoch_log]
    print(f"[train] {[e['line'] for e in epoch_log]}", flush=True)
    print(f"[probe] peak_gpu_gib={peak_gib:.1f} sec_per_step={(wall - (first_step[0] - t0)) / max(1, steps_per_epoch * EPOCHS):.2f}",
          flush=True)

INFO:terra.training.finetune_self_supervised:Loading 1 dataset(s)...
INFO:terra.training.finetune_self_supervised:Loading dataset 1/1: ../../notebooks/in_vivo/terra_pfish/terra_tok


INFO:terra.training.finetune_self_supervised:Loaded 154416 samples
INFO:terra.datasets.dataloaders:Dataloader and -sampler created.
INFO:terra.training.finetune_self_supervised:Initializing encoder, predictor, and target encoder...
INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.
INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          

/data/a330d/projects/terra/src/terra/training/finetune_self_supervised.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(r_path, map_location=torch.devi

INFO:terra.training.finetune_self_supervised:encoder load: <All keys matched successfully>
INFO:terra.training.finetune_self_supervised:predictor load: <All keys matched successfully>
INFO:terra.training.finetune_self_supervised:target_encoder load: <All keys matched successfully>
INFO:terra.training.finetune_self_supervised:Applying LoRA adapters
INFO:terra.training.finetune_self_supervised:Trainable params - encoder: 1191952, predictor: 589824
INFO:terra.utils.helper:Initializing optimizer: AdamW.
INFO:terra.utils.helper:Initializing learning rate scheduler: WarmupCosineSchedule.
INFO:terra.utils.helper:Initializing weight decay scheduler: CosineWDSchedule.
INFO:terra.utils.helper:Initializing automatic mixed precision training scaler: GradScaler.
INFO:terra.training.finetune_self_supervised:Optimizer trainable parameters: 1781776
INFO:terra.training.finetune_self_supervised:Starting I-JEPA self-supervised fine-tuning...
INFO:terra.training.finetune_self_supervised:Start LR: 1e-05, M

  0%|          | 0/1206 [00:00<?, ?it/s]

...

  0%|          | 0/1206 [00:00<?, ?it/s]

INFO:terra.training.finetune_self_supervised:Epoch 5, Batch 1: Grad norm - Encoder: 0.2392, Predictor: 0.0475
INFO:terra.training.finetune_self_supervised:Current LR: 0.000037, WD: 0.296730


100%|██████████| 1206/1206 [12:12<00:00,  1.65it/s]

INFO:terra.training.finetune_self_supervised:Epoch [5/5], Loss: 0.2258, LR: 0.000019


INFO:terra.training.finetune_self_supervised:Checkpoint saved to ../../notebooks/in_vivo/terra_pfish/terra/lora_run/run/checkpoint_epoch_5.pt
INFO:terra.training.finetune_self_supervised:Checkpoint saved to ../../notebooks/in_vivo/terra_pfish/terra/lora_run/run/checkpoint_epoch_5.pt
INFO:terra.training.finetune_self_supervised:Fine-tuning complete!
[train] ['Epoch [1/5], Loss: 0.2455, LR: 0.000095', 'Epoch [2/5], Loss: 0.2275, LR: 0.000080', 'Epoch [3/5], Loss: 0.2248, LR: 0.000059', 'Epoch [4/5], Loss: 0.2253, LR: 0.000037', 'Epoch [5/5], Loss: 0.2258, LR: 0.000019']
[probe] peak_gpu_gib=15.9 sec_per_step=0.61


In [14]:
ckpts = {int(f.stem.split("_")[-1]): f for f in run_dir.glob("checkpoint_epoch_*.pt")}
assert ckpts, f"no checkpoint_epoch_*.pt in {run_dir}"
FINAL_EPOCH = max(ckpts)
print(f"[epochs] checkpoints {sorted(ckpts)} in {run_dir}", flush=True)

(RUN_DIR / "ft_config.json").write_text(json.dumps(
    {"sid": SID, "n_cells": N_CELLS, "epochs": EPOCHS, "max_steps": MAX_STEPS, "batch_size": BATCH_SIZE,
     "lr": LR, "steps_per_epoch": steps_per_epoch, "finetune_dataset": str(FT_TOK),
     "run_dir": str(run_dir), "terra_args": finetune_args}, indent=2, default=str))

[epochs] checkpoints [1, 2, 3, 4, 5] in ../../notebooks/in_vivo/terra_pfish/terra/lora_run/run


1427

### Per-epoch: repack -> verify -> embed every cell -> collapse-guard stats

In [15]:
PRE_ONLINE = torch.load(MODEL_DIR / "model_checkpoint.pt", map_location="cpu")["encoder"]
PRE_ONLINE = {k.replace("module.", ""): v for k, v in PRE_ONLINE.items()}


def repack(epoch, bundle):
    """prepare_finetuned_model for one epoch + the structural checks (these still assert:
    they are correctness, not collapse)."""
    if bundle.exists():
        shutil.rmtree(bundle)
    prepare_finetuned_model(finetuned_checkpoint_dir=str(run_dir), pretrained_model_dir=str(MODEL_DIR),
                            output_dir=str(bundle), checkpoint_epoch=epoch, use_peft=True)
    for f in ("model_config.yaml", "token_dictionary.pkl", "ensembl_dictionary.pkl",
              "gene_count_dictionary.pkl", "model_checkpoint.pt"):
        assert (bundle / f).exists(), f"bundle is missing {f}"
    new_sd = torch.load(bundle / "model_checkpoint.pt", map_location="cpu")["target_encoder"]
    new_sd = {k.replace("module.", ""): v for k, v in new_sd.items()}
    build_encoder().load_state_dict(new_sd)     # strict; raises on any mismatch
    assert set(new_sd) == set(PRETRAINED), "repacked key set differs from pretrained"

    deltas = {k: (new_sd[k].float() - PRETRAINED[k].float()).abs().max().item()
              for k in PRETRAINED if PRETRAINED[k].is_floating_point()}
    changed = sorted(k for k, v in deltas.items() if v > EPS)
    lora_changed = [k for k in changed if any(t in k for t in TARGETS)]
    off_target = [k for k in changed if k not in lora_changed]
    assert lora_changed, "no LoRA-target tensor changed -- fine-tuning had no effect"
    per_block = {b: sum(f".blocks.{b}." in k for k in lora_changed)
                 for b in range(CFG["meta"]["enc_depth"])}
    assert not [b for b, n in per_block.items() if n == 0], f"empty LoRA blocks: {per_block}"
    # Off-target tensors move too, and it is NOT round-off: the exported target encoder is an
    # EMA of the ONLINE encoder, which starts from checkpoint['encoder'] -- a different tensor
    # set from checkpoint['target_encoder'] in the pretrained bundle.  So every frozen weight
    # drifts along that pretrained online/EMA gap and can move no further than it.
    gap = {k: (PRE_ONLINE[k].float() - PRETRAINED[k].float()).abs().max().item() for k in deltas}
    floor = {k: 1e-3 * PRETRAINED[k].float().abs().max().item() for k in deltas}
    unexplained = [k for k in off_target if deltas[k] > 1.01 * gap[k] + max(EPS, floor[k])]
    assert not unexplained, f"off-target tensors moved beyond the pretrained EMA gap: {unexplained[:5]}"
    drift = max((deltas[k] / max(gap[k], 1e-12) for k in off_target), default=0.0)
    print(f"[ep{epoch}] repack {bundle} | {len(changed)}/{len(deltas)} tensors changed: "
          f"{len(lora_changed)} LoRA targets, per-block {per_block}; {len(off_target)} frozen "
          f"tensors EMA-drifted at most {drift:.1%} of the pretrained encoder/target gap", flush=True)
    return {"n_changed_tensors": len(changed), "n_lora_changed": len(lora_changed),
            "lora_changed_per_block": per_block, "n_off_target_drifted": len(off_target),
            "max_off_target_drift_frac_of_ema_gap": drift}


def rank_std(x):
    """(effective rank = exp entropy of the PCA spectrum, mean per-dim std, mean cell-cell cosine)"""
    xc = x - x.mean(0)
    s_ = np.linalg.svd(xc, compute_uv=False) ** 2
    p_ = s_ / s_.sum()
    eff_rank = float(np.exp(-(p_ * np.log(p_ + 1e-12)).sum()))
    xn = x / np.linalg.norm(x, axis=1, keepdims=True)
    cc = xn @ xn.T
    return eff_rank, float(xc.std(0).mean()), float(cc[np.triu_indices(len(x), 1)].mean())

/tmp/ipykernel_1123859/3005243182.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  PRE_ONLINE = torch.load(MODEL_DIR / "model_checkpoint.pt", map_location="cpu")["encoder

In [16]:
# The guard compares the SAME fixed subsample of cells frozen vs fine-tuned; frozen is embedded once.
rng_ = np.random.default_rng(SEED)
GUARD_ROWS = np.sort(rng_.choice(N_CELLS, min(GUARD_CELLS, N_CELLS), replace=False))
tok_guard = tok_all.select(GUARD_ROWS)
frozen = embed_dataset(dataset=tok_guard, model_folder_path=str(MODEL_DIR),
                       **dict(common.EMB_KWARGS, num_workers=4))
frozen = {k: np.asarray(frozen[k], dtype=np.float64) for k in common.EMB_KEYS}

CRITERIA = {"eff_rank": "> 0.5x frozen", "mean_dim_std": "> 0.5x frozen",
            "mean_cosine_to_frozen": "> 0.5"}
epochs = {}
for ep in sorted(ckpts):
    ep_dir = WORK / f"lora_ep{ep}"
    stats = {"repack": repack(ep, ep_dir / "lora_bundle")}
    emb = embed_dataset(dataset=tok_guard, model_folder_path=str(ep_dir / "lora_bundle"),
                        **dict(common.EMB_KWARGS, num_workers=4))
    for k in common.EMB_KEYS:
        a_, b_ = frozen[k], np.asarray(emb[k], dtype=np.float64)
        ok = np.isfinite(b_).all()
        cos = float(np.mean((a_ * b_).sum(1) / (np.linalg.norm(a_, axis=1) * np.linalg.norm(b_, axis=1))))
        mx = float(np.abs(a_ - b_).max())
        (r0, s0, c0), (r1, s1, c1) = rank_std(a_), rank_std(b_)
        stats[k] = {"finite": bool(ok), "max_abs_delta": mx, "mean_cosine_to_frozen": cos,
                    "eff_rank": [r0, r1], "mean_dim_std": [s0, s1],
                    "mean_cell_cell_cosine": [c0, c1],
                    "pass": {"eff_rank": bool(r1 > 0.5 * r0), "mean_dim_std": bool(s1 > 0.5 * s0),
                             "mean_cosine_to_frozen": bool(cos > 0.5)}}
        print(f"[ep{ep}] {k}: max|d| {mx:.4e} | cos-to-frozen {cos:.4f} | eff rank {r0:.1f}->{r1:.1f} "
              f"| dim std {s0:.4f}->{s1:.4f} | cell-cell cos {c0:.3f}->{c1:.3f}", flush=True)
    stats["passed"] = bool(all(v for k in common.EMB_KEYS for v in stats[k]["pass"].values())
                           and all(stats[k]["finite"] for k in common.EMB_KEYS))
    print(f"[ep{ep}] guard passed={stats['passed']}", flush=True)
    epochs[str(ep)] = stats
    del emb

passing = [e for e in sorted(ckpts) if epochs[str(e)]["passed"]]

INFO:terra.inference.embed:STEP 1: LOADING CONFIG...
INFO:terra.inference.embed:STEP 2: GENERATING EMBEDDINGS...
INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.
INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2):

/data/a330d/projects/terra/src/terra/utils/helper.py:455: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


INFO:terra.utils.helper:Loaded pretrained target encoder from epoch 3 with msg: <All keys matched successfully>.
INFO:terra.utils.helper:Finished loading checkpoint with read path: /data/a330d/projects/cellina-reproducibility/pretrained/model_checkpoint.pt.


0it [00:00, ?it/s]/data/a330d/projects/terra/src/terra/inference/embed.py:265: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
157it [02:45,  1.06s/it]

INFO:terra.training.finetune_self_supervised:Loading finetuned checkpoint: ../../notebooks/in_vivo/terra_pfish/terra/lora_run/run/checkpoint_epoch_1.pt


...

[ep3] cell_emb: max|d| 2.5824e+00 | cos-to-frozen 0.8526 | eff rank 31.1->16.7 | dim std 0.4428->0.4476 | cell-cell cos 0.606->0.646
[ep3] spatial_cell_emb: max|d| 1.9802e+00 | cos-to-frozen 0.8722 | eff rank 21.7->12.4 | dim std 0.3040->0.3839 | cell-cell cos 0.802->0.737
[ep3] neighborhood_emb: max|d| 1.8721e+00 | cos-to-frozen 0.8709 | eff rank 19.5->10.3 | dim std 0.2735->0.3564 | cell-cell cos 0.832->0.760
[ep3] guard passed=True
INFO:terra.training.finetune_self_supervised:Loading finetuned checkpoint: ../../notebooks/in_vivo/terra_pfish/terra/lora_run/run/checkpoint_epoch_4.pt
INFO:terra.training.finetune_self_supervised:Merging LoRA adapters into the base model...
INFO:terra.training.finetune_self_supervised:Merged 50 LoRA adapter layers (152 total layers)
INFO:terra.training.finetune_self_supervised:Merged 48 LoRA adapter layers (154 total layers)
INFO:terra.training.finetune_self_supervised:Prepared embeddable model at ../../notebooks/in_vivo/terra_pfish/terra/lora_ep4/lora_b

157it [03:01,  1.16s/it]


[ep4] cell_emb: max|d| 2.7470e+00 | cos-to-frozen 0.8310 | eff rank 31.1->16.0 | dim std 0.4428->0.4539 | cell-cell cos 0.606->0.637
[ep4] spatial_cell_emb: max|d| 2.1109e+00 | cos-to-frozen 0.8526 | eff rank 21.7->12.2 | dim std 0.3040->0.3968 | cell-cell cos 0.802->0.719
[ep4] neighborhood_emb: max|d| 2.0021e+00 | cos-to-frozen 0.8508 | eff rank 19.5->10.1 | dim std 0.2735->0.3687 | cell-cell cos 0.832->0.743
[ep4] guard passed=True
INFO:terra.training.finetune_self_supervised:Loading finetuned checkpoint: ../../notebooks/in_vivo/terra_pfish/terra/lora_run/run/checkpoint_epoch_5.pt
INFO:terra.training.finetune_self_supervised:Merging LoRA adapters into the base model...
INFO:terra.training.finetune_self_supervised:Merged 50 LoRA adapter layers (152 total layers)
INFO:terra.training.finetune_self_supervised:Merged 48 LoRA adapter layers (154 total layers)
INFO:terra.training.finetune_self_supervised:Prepared embeddable model at ../../notebooks/in_vivo/terra_pfish/terra/lora_ep5/lora_b

157it [02:57,  1.13s/it]


[ep5] cell_emb: max|d| 2.7910e+00 | cos-to-frozen 0.8243 | eff rank 31.1->15.8 | dim std 0.4428->0.4559 | cell-cell cos 0.606->0.634
[ep5] spatial_cell_emb: max|d| 2.1470e+00 | cos-to-frozen 0.8464 | eff rank 21.7->12.1 | dim std 0.3040->0.4005 | cell-cell cos 0.802->0.713
[ep5] neighborhood_emb: max|d| 2.0337e+00 | cos-to-frozen 0.8444 | eff rank 19.5->10.1 | dim std 0.2735->0.3722 | cell-cell cos 0.832->0.738
[ep5] guard passed=True


In [17]:
# epoch_selection.json -- mirrors the CRC queue's format
SELECTION.write_text(json.dumps({
    "sid": SID, "adata_path": ADATA_PATH, "run_dir": str(run_dir),
    "final_epoch": FINAL_EPOCH, "latest_passing_epoch": (max(passing) if passing else None),
    "criteria": CRITERIA, "guard_cells": int(len(GUARD_ROWS)), "n_cells": N_CELLS, "steps_per_epoch": steps_per_epoch,
    "loss_per_epoch": loss_per_epoch, "wall_seconds": wall, "peak_gpu_gib": peak_gib,
    "lr": LR, "batch_size": BATCH_SIZE, "epochs": epochs,
    "model_repo": MODEL_REPO, "model_dir": str(MODEL_DIR), "px_to_um": PX_TO_UM,
}, indent=2))
print(json.dumps({"final_epoch": FINAL_EPOCH,
                  "latest_passing_epoch": (max(passing) if passing else None),
                  "passed": {e: epochs[e]["passed"] for e in epochs},
                  "loss_per_epoch": loss_per_epoch}, indent=2), flush=True)
print("wrote", SELECTION, flush=True)

{
  "final_epoch": 5,
  "latest_passing_epoch": 5,
  "passed": {
    "1": true,
    "2": true,
    "3": true,
    "4": true,
    "5": true
  },
  "loss_per_epoch": [
    0.2455,
    0.2275,
    0.2248,
    0.2253,
    0.2258
  ]
}
wrote ../../notebooks/in_vivo/terra_pfish/terra/epoch_selection.json


## 3. Decoder training -- 154 genes, `pfish_analysis.ipynb`'s T-cell holdout split

Mirrors `scripts/terra/inference.py`'s `train_decoder()` (decoder track): embeddings from the
**best fine-tuned encoder** (`latest_passing_epoch` from `epoch_selection.json` -- epoch 5, since
all 5 passed the collapse guard) -> a small count-decoder head (`terra.training.decode`, same CLI
as CRC: `nb_libsize` loss, hidden-dim 512, 2-layer MLP, layer norm, early-stop patience 5) trained
to predict the **154 genes in `pfish_full.h5ad`** (not the 500-gene encoder panel).

**Split**: identical to `pfish_analysis.ipynb` (cells 1/3/9) -- `pfish_prep.make_edge_swap_sets`'s
`idx_target` (T cells with >=1 perturbed-cancer spatial neighbour, Gaussian-kernel graph
bandwidth=250, max 30 neighbours) is held out of train **and** val entirely, matching the encoder's
own "encoder sees everything, only the readout is held out" asymmetry from CRC. Everything else is
split 90/10 train/val (`train_test_split(..., seed=0)`). Since `cellina` (which this split logic
imports) isn't installed in the `terra` conda env, the split was precomputed once in `cellina-graph`
via `notebooks/in_vivo/make_decoder_split.py` and saved as cell-id sets (not positional indices, so
it's robust to the different cell filters `harmonize_adata` and `pfish_prep.filter_cells` each apply)
to `notebooks/in_vivo/terra_pfish/decoder_split.json`.

In [12]:
import pickle as _pkl

with open("../../notebooks/in_vivo/terra_pfish/terra/epoch_selection.json") as f:
    _sel = json.load(f)
BEST_EPOCH = _sel["latest_passing_epoch"]
assert BEST_EPOCH is not None, "no epoch passed the collapse guard"
print(f"[decoder] using lora_ep{BEST_EPOCH} (final_epoch={_sel['final_epoch']})")

DEC_ENC_DIR = WORK / f"lora_ep{BEST_EPOCH}" / "lora_bundle"
assert DEC_ENC_DIR.exists(), f"missing {DEC_ENC_DIR} -- rerun section 2's guard loop"

with open("../../notebooks/in_vivo/terra_pfish/decoder_split.json") as f:
    split = json.load(f)
print({k: len(v) for k, v in split.items() if k != "meta"}, split["meta"])

[decoder] using lora_ep5 (final_epoch=5)
{'train_ids': 118606, 'val_ids': 13179, 'test_ids': 22326, 'control_ids': 14674} {'seed': 0, 'bandwidth': 250.0, 'max_neighbours': 30, 'h5': '/data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/pfish_full.h5ad', 'n_obs_after_filter_cells': 154111}


In [13]:
# --- embed all harmonized cells through the best LoRA epoch (cached, same pattern as common.embed_cached) ---
emb, ids = common.embed_cached(tok_all, DEC_ENC_DIR, WORK / f"emb_lora_ep{BEST_EPOCH}.npz")
# cellina's readout: cat(z from the cell's own counts, s from its neighbourhood)
scemb = np.hstack([np.asarray(emb["cell_emb"]), np.asarray(emb["neighborhood_emb"])]).astype(np.float32)
assert np.isfinite(scemb).all(), "cell_emb/neighborhood_emb: non-finite rows"
print(f"[decoder] embedded {scemb.shape[0]:,} cells -> {scemb.shape[1]}-d readout")

# --- 154-gene ground truth from pfish_full.h5ad, aligned to the embedding's cell_id order ---
with h5py.File(PFISH_FULL, "r") as f:
    full_ids = [g.decode() if isinstance(g, bytes) else g for g in f["obs"]["_index"][:]]
    full_X = f["X"][:]                     # (154418, 154) int32 raw counts, columns == genes_154 order
full_pos = {c: i for i, c in enumerate(full_ids)}
counts_154 = full_X[[full_pos[c] for c in ids]]
print(f"[decoder] ground truth {counts_154.shape} ({len(genes_154)} genes, order matches genes_154)")

# --- map the pfish_analysis.ipynb split (cell-id sets) onto this embedding's row order ---
pos_of = {c: i for i, c in enumerate(ids)}


def _to_idx(id_list):
    found = [pos_of[c] for c in id_list if c in pos_of]
    return np.array(found, dtype=np.int64), len(id_list) - len(found)


train_idx, n_drop_tr = _to_idx(split["train_ids"])
val_idx, n_drop_va = _to_idx(split["val_ids"])
test_idx, n_drop_te = _to_idx(split["test_ids"])
print(f"[decoder] train={len(train_idx)} (dropped {n_drop_tr}) | val={len(val_idx)} (dropped {n_drop_va}) "
      f"| test={len(test_idx)} (dropped {n_drop_te}) -- drops are harmonize_adata's 2 filtered-out cells")
assert not (set(train_idx.tolist()) & set(val_idx.tolist())), "train/val overlap"
assert not (set(train_idx.tolist()) & set(test_idx.tolist())), "train/test overlap -- held-out T cells leaked"
assert not (set(val_idx.tolist()) & set(test_idx.tolist())), "val/test overlap -- held-out T cells leaked"

loaded cached embeddings /data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/terra_pfish/terra/emb_lora_ep5.npz
[decoder] embedded 154,416 cells -> 768-d readout
[decoder] ground truth (154416, 154) (154 genes, order matches genes_154)
[decoder] train=118604 (dropped 2) | val=13179 (dropped 0) | test=22326 (dropped 0) -- drops are harmonize_adata's 2 filtered-out cells


In [11]:
import subprocess

# --- assemble + run terra.training.decode -- same CLI/hyperparameters as inference.py's train_decoder() ---
DECODER_EPOCHS = 100
VARIANT = f"lora_ep{BEST_EPOCH}"
_REPO = Path.cwd().parents[1]

ckpt = WORK / f"count_decoder_{VARIANT}.pt"
metrics_path = WORK / f"decoder_metrics_{VARIANT}.json"
npz_path = WORK / f"decoder_data_{VARIANT}.npz"

pack = {}
for name, idx, expr_key in (("train", train_idx, "train_expression"),
                             ("val", val_idx, "val_expression"),
                             ("test", test_idx, "test_expression_gt")):
    pack[f"{name}_embeddings"] = scemb[idx]
    pack[expr_key] = counts_154[idx].astype(np.float32)
    pack[f"{name}_barcodes"] = np.array([ids[i] for i in idx], dtype=object)
    pack[f"{name}_slides"] = np.full(len(idx), SID, dtype=object)

np.savez(npz_path, gene_list=np.array(genes_154, dtype=object), **pack)
del pack

cmd = [sys.executable, "-m", "terra.training.decode",
       "--dataset", str(npz_path), "--gene-selection", "all",
       "--loss-type", "nb_libsize", "--disable-slide-batching",
       "--epochs", str(DECODER_EPOCHS), "--hidden-dim", "512", "--mlp-depth", "2", "--layer-norm",
       "--early-stop-patience", "5", "--device", "0", "--seed", str(SEED),
       "--output", str(ckpt), "--metrics-json", str(metrics_path)]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(_REPO))
npz_path.unlink()   # embeddings + counts NPZ can be rebuilt from the cells above on demand

/data/a330d/miniforge3/envs/terra/bin/python -m terra.training.decode --dataset /data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/terra_pfish/terra/decoder_data_lora_ep5.npz --gene-selection all --loss-type nb_libsize --disable-slide-batching --epochs 100 --hidden-dim 512 --mlp-depth 2 --layer-norm --early-stop-patience 5 --device 0 --seed 0 --output /data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/terra_pfish/terra/count_decoder_lora_ep5.pt --metrics-json /data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/terra_pfish/terra/decoder_metrics_lora_ep5.json


/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/data/a330d/miniforge3/envs/terra/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


INFO:__main__:Using loss type: nb_libsize
INFO:__main__:Train library size: mean=221.05, std=149.84, min=40.00, max=800.00
INFO:__main__:Test library size: mean=181.80, std=115.19, min=40.00, max=800.00
INFO:__main__:Val library size: mean=221.22, std=151.67, min=40.00, max=800.00
INFO:__main__:Using fixed library size of 10000.0 at test time
INFO:__main__:[info] Initial metrics (epoch 0, untrained): train pearson_mean=-0.0093, val pearson_mean=-0.0088, test pearson_mean=-0.0097
INFO:__main__:Early stopping at epoch 25
INFO:__main__:Saved checkpoint to: /data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/terra_pfish/terra/count_decoder_lora_ep5.pt
INFO:__main__:Loss type: nb_libsize
INFO:__main__:Best val Pearson: 0.5382
INFO:__main__:Best test Pearson: 0.5730


In [12]:
# --- sanity: the checkpoint's own gene_list matches genes_154, and val/test pearson aren't identical
# (which would mean the decoder is silently selecting on the held-out set) ---
gene_list = list(torch.load(ckpt, map_location="cpu")["gene_list"])
assert gene_list == genes_154, "decoder gene_list != genes_154"
m = json.loads(metrics_path.read_text())
assert abs(m["val"]["pearson_mean"] - m["test"]["pearson_mean"]) > 1e-12, (
    "val.pearson_mean == test.pearson_mean -- suspicious, check the split")
print(f"[{VARIANT}] decoder val pearson {m['val']['pearson_mean']:.4f} | "
      f"test pearson (held-out perturbed-niche T cells) {m['test']['pearson_mean']:.4f}")

[lora_ep5] decoder val pearson 0.5382 | test pearson (held-out perturbed-niche T cells) 0.5730


/tmp/ipykernel_1244767/3696915084.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gene_list = list(torch.load(ckpt, map_location="cpu")["gene_list"])


## 4. Counterfactual evaluation -- `pfish_analysis.ipynb`'s effective-KO analysis, for TERRA

Same eval statistics as `pfish_analysis.ipynb` section 4 (`_metrics_row`, `scores()`, the
real/mean/random comparison, `results/invivo_{VARIANT}.csv`), reused essentially verbatim. The only
thing that changes is **how the counterfactual is produced**:

- **Cellina's `insert(sig)`**: overwrite a far T cell's precomputed niche feature `spatial_x` with a
  KO signature vector, decode with Cellina's own NB head.
- **TERRA's equivalent**: build one synthetic "average KO cancer cell" (raw counts recovered from
  the KO's mean **lognorm** profile -- `sc.pp.normalize_total(target_sum=None)` + `log1p`, i.e.
  `pfish_terra.h5ad`'s own `layers['lognorm']`, matching `pfish_analysis.ipynb`'s `ln[kc].mean(0)`
  exactly, not a 1e4-CPM shortcut), tokenize it through TERRA's own tokenizer, then **replace all 10
  neighbourhood token blocks** of each far T cell with 10 copies of that synthetic cell -- the
  token-level analogue of "as if the niche were 100% perturbed cancer". The far cell's own 256-token
  cell segment (identity `z`) is left untouched, exactly like `dc==0` in `scripts/terra/inference.py`'s
  `perturb()` and like Cellina keeping `z` fixed.
- `z` (`cell_emb`) comes from the **unperturbed** embedding pass, `s` (`neighborhood_emb`) from the
  **perturbed** one -- `cat(z, s)` -> the section-3 decoder -> predicted counterfactual counts.

Everything data-side (`OBS`, `mean_shift`, `shared`, `recon`, `far`) is unchanged from
`pfish_analysis.ipynb` -- precomputed once in the `cellina-graph` env (needs `cellina`'s spatial
graph) via `notebooks/in_vivo/make_counterfactual_ground_truth.py` and loaded here as
`counterfactual_ground_truth.npz`.

In [ ]:
import shutil

import anndata as ad
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, wilcoxon

from terra.inference import tokenize_adata
from terra.training.decode import apply_count_decoder

gt = np.load("../../notebooks/in_vivo/terra_pfish/counterfactual_ground_truth.npz", allow_pickle=True)

genes_154_gt = list(gt["genes_154"])
assert genes_154_gt == genes_154, "counterfactual ground truth is on a different gene order"
far_ids_all = list(gt["far_ids"])
recon = gt["recon"]                     # (6000, 154), 1e4-CPM normalized real far-cell baseline
p_recon = recon.mean(0)
mean_shift = gt["mean_shift"]
shared = gt["shared"]
EFF = list(gt["EFF9"])
OBS = {ko: gt[f"OBS_{ko}"] for ko in EFF}
print(f"[cf] far={len(far_ids_all)} EFF={EFF}")

# --- 500-gene lognorm KO signatures, straight from pfish_terra.h5ad's own layer/obs (no cellina needed) ---
# depth must come from THIS 500-gene panel's own n_counts, not counterfactual_ground_truth's 154-gene
# DEPTH -- random_sigs below already uses the 500-gene total, so mixing panels made synthetic KO
# neighbours systematically shallower than random ones (a depth confound, since TERRA reads raw counts).
adata_terra_full = sc.read_h5ad(PFISH_TERRA)
ln500 = np.asarray(adata_terra_full.layers["lognorm"])
obsT = adata_terra_full.obs
is_cancer = (obsT["celltype2"] == "cancer").to_numpy()
pert_col = obsT["perturbation"].astype(str).to_numpy()
n_perturb = obsT["n_perturb"].to_numpy()
pc = is_cancer & (n_perturb > 0)
n_counts500 = obsT["n_counts"].to_numpy()


def ko_signature(ko):
    kc = pc & (pert_col == ko)
    return ln500[kc].mean(0), float(n_counts500[kc].mean())


rng = np.random.default_rng(SEED)
cancer_idx = np.where(is_cancer)[0]
N_RANDOM = 5
random_sigs = []
for _ in range(N_RANDOM):
    draw = rng.choice(cancer_idx, 300, replace=False)
    random_sigs.append((ln500[draw].mean(0), float(obsT["n_counts"].to_numpy()[draw].mean())))

del adata_terra_full

In [48]:
len(SIGNATURES) * CLUSTER_N

154

In [ ]:
# --- build one 11-cell synthetic constellation per signature (9 KO + 5 random draws), clusters
# spaced 10,000um apart so TERRA's own n_neighs=10 KNN never crosses between them ---
SIGNATURES = {ko: ko_signature(ko) for ko in EFF}
SIGNATURES.update({f"random_{i}": random_sigs[i] for i in range(N_RANDOM)})

CLUSTER_N = 11     # 1 "self" + 10 identical neighbours per cluster
CLUSTER_SPACING = 10_000.0   # um
JITTER = 2.0                 # um -- avoids zero-distance ties in the tokenizer's KNN
SEQ_LEN_CELL = 256

rows, coords, obs_names_syn = [], [], []
rng_syn = np.random.default_rng(SEED)
for ci, (name, (sig, depth)) in enumerate(SIGNATURES.items()):
    sig_prop = np.expm1(sig)
    sig_prop = sig_prop / (sig_prop.sum() + 1e-8)
    synth_counts = np.round(sig_prop * depth).astype(np.int32)
    center = np.array([ci * CLUSTER_SPACING, 0.0])
    for j in range(CLUSTER_N):
        rows.append(synth_counts)
        coords.append(center + rng_syn.uniform(-JITTER, JITTER, size=2))
        obs_names_syn.append(f"syn_{name}_{j}")

syn = ad.AnnData(X=np.stack(rows).astype(np.int32), var=pd.DataFrame(index=genes_500))
syn.obs_names = obs_names_syn
syn.obs["cell_id"] = syn.obs_names
syn.obsm["spatial"] = np.asarray(coords)
syn.layers["counts"] = syn.X.copy()

syn_h = harmonize_adata(syn, gene_mapping_dict_file_path=f"{MODEL_DIR}/ensembl_dictionary.pkl",
                        gene_occurrence_count_file_path=f"{MODEL_DIR}/gene_count_dictionary.pkl",
                        min_genes_per_cell=1, min_cells_per_gene=1)
print(f"[synth] {len(SIGNATURES)} signatures x {CLUSTER_N} cells -> harmonized "
      f"{syn_h.n_obs} cells x {syn_h.n_vars} genes")

_tmp_tok = WORK / ".scratch_syn_tok_tmp"
shutil.rmtree(_tmp_tok, ignore_errors=True)
syn_tok = tokenize_adata(syn_h, str(MODEL_DIR), str(_tmp_tok), nproc=4)
shutil.rmtree(_tmp_tok, ignore_errors=True)
syn_raw = syn_tok.with_format(None)
syn_ids = [str(c) for c in syn_raw["cell_id"]]

# row 0 of each cluster -> its own 256-token cell segment == the synthetic cell's representation
NEIGH_BLOCK = {}
for name in SIGNATURES:
    pos = syn_ids.index(f"syn_{name}_0")
    gt_tok = np.asarray(syn_raw["gene_tokens"][pos])[:SEQ_LEN_CELL]
    ge_tok = np.asarray(syn_raw["gene_expr"][pos])[:SEQ_LEN_CELL]
    NEIGH_BLOCK[name] = (gt_tok, ge_tok)
    print(f"  {name}: depth={SIGNATURES[name][1]:.1f} nonzero_tokens={(gt_tok != 0).sum()}")

In [14]:
# --- far cells: same 6000-cell subsample as pfish_analysis.ipynb, restricted to what survived
# harmonize_adata (drops at most the 2 cells dropped in section 2) ---
pos_of_all = {c: i for i, c in enumerate(ids)}     # `ids` == tok_all's cell_id order, from section 3
keep = [c in pos_of_all for c in far_ids_all]
n_drop_far = len(far_ids_all) - sum(keep)
far_ids = [c for c, k in zip(far_ids_all, keep) if k]
recon_far = recon[np.array(keep)]
p_recon = recon_far.mean(0)
far_rows = [pos_of_all[c] for c in far_ids]
tok_far = tok_all.select(far_rows)
print(f"[far] {len(far_ids)} cells (dropped {n_drop_far} not in the harmonized panel)")

# z (cell_emb) is always the UNPERTURBED pass -- identity stays fixed, same as Cellina/inference.py
z_emb = embed_dataset(dataset=tok_far, model_folder_path=str(DEC_ENC_DIR), **dict(common.EMB_KWARGS, num_workers=4))
z = np.asarray(z_emb["cell_emb"], dtype=np.float32)
print(f"[far] z (unperturbed cell_emb) {z.shape}")

[far] 6000 cells (dropped 0 not in the harmonized panel)
INFO:terra.inference.embed:STEP 1: LOADING CONFIG...
INFO:terra.inference.embed:STEP 2: GENERATING EMBEDDINGS...
INFO:terra.utils.helper:Protein-init: DISABLED -- using the default learnable nn.Embedding for gene tokens.


INFO:terra.utils.helper:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
    (pos_embed): Embedding(2

/data/a330d/projects/terra/src/terra/utils/helper.py:455: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(
0it [00:00, ?it/s]/data/a330d/projects/terra

[far] z (unperturbed cell_emb) (6000, 384)


In [18]:
VARIANT = f"lora_ep{BEST_EPOCH}"
ckpt = WORK / f"count_decoder_{VARIANT}.pt"

In [ ]:
recon = recon_far   # rename to match pfish_analysis.ipynb's eval cells below


def _normalize_counts(x, eps=1e-8, scale=1e4):
    return x / (x.sum(axis=1, keepdims=True) + eps) * scale


def lfc(a, b):
    return np.log(a + 1) - np.log(b + 1)


def _splice_neighbourhood(batch, gt_block, ge_block):
    """Replace all 10 neighbourhood blocks with 10 copies of the synthetic cell; the far cell's
    own 256-token cell segment (z) is left untouched -- mirrors inference.py's `dc==0` invariant."""
    tokens = np.asarray(batch["gene_tokens"])
    expr = np.asarray(batch["gene_expr"], dtype=np.float32)
    tokens[:, SEQ_LEN_CELL:] = np.tile(gt_block, 10)[None, :]
    expr[:, SEQ_LEN_CELL:] = np.tile(ge_block, 10)[None, :]
    batch["gene_tokens"], batch["gene_expr"] = tokens, expr
    return batch


def terra_insert(name):
    """TERRA's equivalent of pfish_analysis.ipynb's `insert(sig)`: saturate the niche with the
    signature `name`, decode, and return (lfc_vs_baseline, predicted_counts_1e4_normalized)."""
    gt_block, ge_block = NEIGH_BLOCK[name]
    # with_format(None) for the .map(), then restore -- embed_dataset's DataLoader needs
    # rel_x_coord/rel_y_coord etc. back in their original (tensor-ready) format, same fix as
    # common.map_perturbation uses for the CRC neighbour-perturbation pipeline.
    fmt = tok_far.format
    pert_tok = tok_far.with_format(None).map(
        lambda b: _splice_neighbourhood(b, gt_block, ge_block),
        batched=True, batch_size=256, keep_in_memory=True, load_from_cache_file=False)
    pert_tok.set_format(type=fmt["type"], columns=fmt["columns"], output_all_columns=fmt["output_all_columns"])
    s_emb = embed_dataset(dataset=pert_tok, model_folder_path=str(DEC_ENC_DIR),
                          **dict(common.EMB_KWARGS, num_workers=4))
    s = np.asarray(s_emb["neighborhood_emb"], dtype=np.float32)
    lat = np.hstack([z, s])

    a_ = ad.AnnData(X=np.zeros((len(lat), len(genes_154)), dtype=np.float32),
                    var=pd.DataFrame(index=pd.Index(genes_154)))
    a_.obsm["decoder_emb"] = lat
    apply_count_decoder(a_, emb_key="decoder_emb", model_folder_path=None, checkpoint_path=str(ckpt),
                        decoded_counts_layer_key="decoded", embed_fallback_key="decoder_emb", device=0)
    pr = _normalize_counts(np.asarray(a_.layers["decoded"]))
    return lfc(pr.mean(0), p_recon), pr


REAL, COUNTERFACTUALS = {}, {}
for ko in EFF:
    REAL[ko], COUNTERFACTUALS[ko] = terra_insert(ko)
    print(f"[{ko}] REAL lfc: mean|lfc|={np.abs(REAL[ko]).mean():.3f} top10-vs-OBS r="
          f"{pearsonr(OBS[ko][np.argsort(-np.abs(OBS[ko]))[:10]], REAL[ko][np.argsort(-np.abs(OBS[ko]))[:10]])[0]:.3f}")

rand, CF_RANDOM = [], []
for i in range(N_RANDOM):
    r_lfc, r_cf = terra_insert(f"random_{i}")
    rand.append(r_lfc)
    CF_RANDOM.append(r_cf)
CF_RANDOM = np.array(CF_RANDOM)

CF_MEAN = np.clip((recon + 1) * np.exp(mean_shift)[None, :] - 1, 0, None)
print("done: REAL/COUNTERFACTUALS for", len(EFF), "KOs, rand/CF_RANDOM for", N_RANDOM, "draws")

In [ ]:
# --- pfish_analysis.ipynb cells 19-21 (final state: EFF9_2=EFF, REAL5=REAL -- no gene subsetting) ---
EFF9_2 = EFF
REAL5 = REAL


def rN(o_, p_, n=10):
    d = np.argsort(-np.abs(o_))[:n]
    return pearsonr(o_[d], p_[d])[0]


def scores(resid, N=10, EFF=EFF):
    tg = {k: (OBS[k] - shared if resid else OBS[k]) for k in EFF}
    re = [rN(tg[k], REAL5[k], N) for k in EFF]
    ra = [np.mean([rN(tg[k], rp, N) for rp in rand]) for k in EFF]
    me = [rN(tg[k], mean_shift, N) for k in EFF]
    return np.array(re), np.array(ra), np.array(me)


C = {"real": "#009E73", "mean": "#E69F00", "random": "#999999"}
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for a, (resid, ttl) in zip(ax, [(False, "Full signal"), (True, "KO-specific component")]):
    re, ra, me = scores(resid, EFF=EFF9_2)
    print(re)
    for j, (v, name) in enumerate([(re, "real"), (me, "mean"), (ra, "random")]):
        a.scatter(np.full(len(v), j) + rng.uniform(-.09, .09, len(v)), v, s=45,
                  color=C[name], edgecolor="k", lw=.4, zorder=3)
        a.hlines(v.mean(), j - .3, j + .3, color=C[name], lw=3, zorder=4)
    for k in range(len(EFF9_2)):
        a.plot([0, 1, 2], [re[k], me[k], ra[k]], color="0.82", lw=.6, zorder=1)
    a.axhline(0, color="0.8", lw=.7)
    a.set_xticks([0, 1, 2])
    a.set_xticklabels(["real", "mean", "random"])
    pr_ = wilcoxon(re, ra, alternative="greater").pvalue
    pm_ = wilcoxon(re, me, alternative="greater").pvalue
    a.set_title(f"{ttl}\nreal>random p={pr_:.3f}   real>mean p={pm_:.3f}", fontsize=10)
ax[0].set_ylabel("top-10 Pearson r (predicted vs observed)")
plt.tight_layout()
plt.show()
for resid, ttl in [(False, "full"), (True, "KO-specific")]:
    re, ra, me = scores(resid, EFF=EFF9_2)
    print(f"{ttl:12s}: real={re.mean():.3f}  mean={me.mean():.3f}  random={ra.mean():.3f}")

In [ ]:
# --- pfish_analysis.ipynb cells 22-27: Pearson / Precision / E-distance / RMSE_LFC per KO ---
sys.path.insert(0, str(Path.cwd().parent))   # repo scripts/ -- for counterfactual_analysis
from counterfactual_analysis import compute_edistance, direction_match, compute_mse_lfc

n_deg = 10
counts_per_k = 1e4
TERRA_VARIANT = f"terra-lora-ep{BEST_EPOCH}"

# adata for compute_edistance(..., use_pca=True): needs var_names==genes_154 + is_holdout (idx_target)
adata_154 = sc.read_h5ad(PFISH_FULL)
adata_154.obs["is_holdout"] = adata_154.obs_names.astype(str).isin(set(split["test_ids"]))
print(f"[edist] is_holdout: {adata_154.obs['is_holdout'].sum()} / {adata_154.n_obs}")

# E-distance ground truth = each KO's exclusive near-KO T cells (held-out targets, pfish_analysis.ipynb
# cell 17's NEAR_EXPR), NOT the far/control `recon` the counterfactuals were predicted from.
# Raw 154-gene counts from pfish_full.h5ad (cell 24's full_X), 1e4-normalized like recon/predictions.
NEAR_EXPR = {}
for ko in EFF:
    near_ids = [str(c) for c in gt[f"NEAR_IDS_{ko}"]]
    assert set(near_ids) <= set(split["test_ids"]), f"{ko}: near-KO T cells outside the held-out split"
    NEAR_EXPR[ko] = _normalize_counts(full_X[[full_pos[c] for c in near_ids]].astype(np.float64))
    print(f"[edist] {ko}: {len(near_ids)} near-KO target T cells")


def _metrics_row(k, lfc_pred, lfc_observed, predicted_expr):
    deg = np.argsort(-np.abs(lfc_observed))[:n_deg]
    pear, _ = pearsonr(lfc_observed[deg], lfc_pred[deg])
    dir_match_k = direction_match(lfc_observed[deg], lfc_pred[deg], k=n_deg, normalize="k")
    mse_lfc = compute_mse_lfc(gt_vec=lfc_observed, cf_vec=lfc_pred, deg=deg)
    rmse_lfc = np.sqrt(mse_lfc)
    edist_pca_log = compute_edistance(adata_154, observed=NEAR_EXPR[k], predicted=predicted_expr, deg=None,
                                      library_size=counts_per_k, local=True, use_pca=True)
    return {"Pearson": pear, "Precision": dir_match_k, "E-distance": edist_pca_log, "RMSE_LFC": rmse_lfc}


full_results = [{"KO": k, **_metrics_row(k, REAL5[k], OBS[k], COUNTERFACTUALS[k])} for k in EFF9_2]
df_results = pd.DataFrame(full_results)
df_results["type"] = "full"
df_results["baseline"] = TERRA_VARIANT

ko_results = pd.DataFrame([{"KO": k, **_metrics_row(k, REAL5[k], OBS[k] - shared, COUNTERFACTUALS[k])} for k in EFF9_2])
ko_results["type"] = "KO-specific"
ko_results["baseline"] = TERRA_VARIANT
df_results = pd.concat([df_results, ko_results], ignore_index=True)

mean_results = []
for k in EFF9_2:
    mean_results.append({"KO": k, "type": "full", "baseline": "mean", **_metrics_row(k, mean_shift, OBS[k], CF_MEAN)})
    mean_results.append({"KO": k, "type": "KO-specific", "baseline": "mean",
                         **_metrics_row(k, mean_shift, OBS[k] - shared, CF_MEAN)})
df_results = pd.concat([df_results, pd.DataFrame(mean_results)], ignore_index=True)

random_results = []
for k in EFF9_2:
    full_draws = pd.DataFrame([_metrics_row(k, rand[i], OBS[k], CF_RANDOM[i]) for i in range(len(rand))])
    ko_draws = pd.DataFrame([_metrics_row(k, rand[i], OBS[k] - shared, CF_RANDOM[i]) for i in range(len(rand))])
    random_results.append({"KO": k, "type": "full", "baseline": f"{TERRA_VARIANT}-random", **full_draws.mean().to_dict()})
    random_results.append({"KO": k, "type": "KO-specific", "baseline": f"{TERRA_VARIANT}-random", **ko_draws.mean().to_dict()})
df_results = pd.concat([df_results, pd.DataFrame(random_results)], ignore_index=True)

metrics = ["Pearson", "Precision", "E-distance", "RMSE_LFC"]
summary_fmt = df_results.groupby(["type", "baseline"])[metrics].agg(lambda x: f"{x.mean():.2f} ± {x.std():.2f}")
print(summary_fmt)

df_results.to_csv("../../results/invivo_terra.csv")
print("wrote ../../results/invivo_terra.csv")
df_results

In [23]:
summary_fmt

Pearson    Precision    E-distance  \
type        baseline                                                         
KO-specific mean                   -0.01 ± 0.31  0.54 ± 0.14  -0.11 ± 0.01   
            terra-lora-ep5          0.15 ± 0.28  0.57 ± 0.17   1.85 ± 0.07   
            terra-lora-ep5-random   0.04 ± 0.39  0.53 ± 0.17   1.81 ± 0.03   
full        mean                    0.37 ± 0.38  0.77 ± 0.17  -0.10 ± 0.01   
            terra-lora-ep5          0.13 ± 0.30  0.52 ± 0.11   1.88 ± 0.07   
            terra-lora-ep5-random  -0.02 ± 0.36  0.47 ± 0.13   1.83 ± 0.03   

                                      RMSE_LFC  
type        baseline                            
KO-specific mean                   0.43 ± 0.14  
            terra-lora-ep5         0.43 ± 0.14  
            terra-lora-ep5-random  0.45 ± 0.15  
full        mean                   0.52 ± 0.15  
            terra-lora-ep5         0.59 ± 0.11  
            terra-lora-ep5-random  0.63 ± 0.12